# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/taqadussana/ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Random Forest Classifier**

My baseline (Week 4) leaned on staleness + visibility combined into one rule, but Section 1
of that notebook showed staleness alone was only **MIXED** as a signal — decline rate didn't
rise cleanly with staleness. That's exactly the kind of messy, multi-signal pattern a single
rule can't capture but a tree-based model can: a Random Forest can combine staleness,
visibility, position, and CTR together and find real interactions between them, rather than
relying on one threshold.

I'm picking Random Forest over a single Decision Tree because my Week 1 experiment already
showed a lone tree tends to overfit to whichever features happen to split first — an ensemble
of trees averages that out and should give a more honest, stable ranking.

**Method: Random Forest Classifier**

My baseline (Week 4) leaned on staleness + visibility combined into one rule, but Section 1
of that notebook showed staleness alone was only **MIXED** as a signal — decline rate didn't
rise cleanly with staleness. That's exactly the kind of messy, multi-signal pattern a single
rule can't capture but a tree-based model can: a Random Forest can combine staleness,
visibility, position, and CTR together and find real interactions between them, rather than
relying on one threshold.

I'm picking Random Forest over a single Decision Tree because my Week 1 experiment already
showed a lone tree tends to overfit to whichever features happen to split first — an ensemble
of trees averages that out and should give a more honest, stable ranking.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split: client-grouped, not random.**

Pages from the same client likely share patterns (same site structure, same content
strategy, same seasonal rhythm). If I split randomly, pages from one client could land in
both train and test — the model would partly be memorizing that client's quirks rather than
learning a generalizable pattern, making my test score look better than it really is.

Using a grouped split (grouping by `client_id`) means entire clients are held out for
testing — the model is evaluated on clients it has never seen a single page from. This is
the same discipline the starter pipeline itself used (client-holdout validation) to get its
reported numbers.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Model vs Baseline — same test split, same metric**

| K  | Baseline | Random Forest |
|----|----------|----------------|
| 20 | 0.200    | **0.550**      |
| 50 | 0.320    | **0.580**      |

I compared my Week 4 baseline (`aging_visible_page` scoring rule) against a Random Forest,
using the exact same client-grouped test split and the exact same Precision@K metric for
both, so this is a fair, apples-to-apples comparison — not two different setups.

The Random Forest beats the baseline by a wide margin at both cutoffs — nearly triple the
precision at K=20 (0.550 vs 0.200), and still well ahead at K=50 (0.580 vs 0.320). This lines
up directly with the permutation importance results in Section 4: the model leans heavily on
`impressions_90d`, while `days_since_last_update` — the core signal my baseline's rule was
built around — turned out to contribute almost nothing on its own. My baseline's real
strength was its visibility half, not the staleness half, and the model independently
confirms that by finding the same pattern without being told it in advance.

This gap is measured on held-out clients the model never trained on (client-grouped split),
so it's a genuine generalization result, not an artifact of memorized clients — though it's
still evidence from one train/test split, not a guarantee that holds across every future
month or every client.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

url = "https://raw.githubusercontent.com/taqadussana/ML/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Rebuild baseline reason/action exactly as in Week 4
def assign_reason(row):
    if 90 <= row["days_since_last_update"] <= 180 and row["impressions_90d"] >= 500:
        return "aging_visible_page"
    elif row["impressions_90d"] >= 500:
        return "high_visibility_monitor"
    else:
        return "low_signal"
df["reason_code"] = df.apply(assign_reason, axis=1)

window_center = 135
df["aging_score"] = 1 - (abs(df["days_since_last_update"] - window_center) / df["days_since_last_update"].max()).clip(0, 1)
df["visibility_score"] = np.log1p(df["impressions_90d"]) / np.log1p(df["impressions_90d"].max())
df["baseline_action_score"] = 0.5 * df["aging_score"] + 0.5 * df["visibility_score"]

# Client-grouped split
features = ["avg_position", "impressions_90d", "sessions_90d", "content_age_days",
            "days_since_last_update", "ctr", "engagement_rate", "word_count"]
features = [f for f in features if f in df.columns]  # keep only columns that actually exist

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
baseline_test = df.iloc[test_idx]["baseline_action_score"].values

rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print("Model vs Baseline — same test split, same metric\n")
print(f"{'K':<6}{'Baseline':<12}{'Random Forest':<15}")
for k in (20, 50):
    b = precision_at_k(baseline_test, y_test.values, k)
    m = precision_at_k(rf_scores, y_test.values, k)
    print(f"{k:<6}{b:<12.3f}{m:<15.3f}")

Model vs Baseline — same test split, same metric

K     Baseline    Random Forest  
20    0.200       0.550          
50    0.320       0.580          


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Permutation importance shows **impressions_90d** mattering most by far (0.057), more than
double the next feature (content_age_days, 0.023). avg_position and ctr contribute modestly
(0.014, 0.012). Strikingly, **days_since_last_update** — the staleness signal my Week 4
baseline leaned on — comes back essentially useless here (-0.014, meaning it doesn't help
and may slightly hurt). This actually confirms what Section 1 of my baseline notebook already
flagged: staleness alone was only MIXED, not a reliable signal, and the model independently
arrived at the same conclusion.

Looking at false positives (1,815) and false negatives (1,230): false positives tend to be
pages with strong recent metrics (e.g. one has 3,998 sessions_90d and only 8 days since
update) that the model still flagged as risky — likely because impressions_90d alone pushed
the score up without enough countervailing signal. False negatives are more concerning: some
have very high impressions_90d (15,320 in one case) yet were missed, suggesting the model
underweights genuinely large but slowly-declining pages, possibly because most training
examples at that traffic level are non-declining.

This is observed, decision-support evidence from one train/test split — not a claim that
this model works for every client or every time period.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

# Permutation importance
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, scoring="average_precision")
importance_df = pd.DataFrame({
    "feature": features,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)
print("Feature importance (permutation):")
print(importance_df.to_string(index=False))

# Look at top false positives and false negatives
test_df = df.iloc[test_idx].copy()
test_df["rf_score"] = rf_scores
test_df["predicted_high_risk"] = test_df["rf_score"] >= 0.5

false_positives = test_df[(test_df["predicted_high_risk"] == True) & (test_df["is_declining_label"] == 0)]
false_negatives = test_df[(test_df["predicted_high_risk"] == False) & (test_df["is_declining_label"] == 1)]

print(f"\nFalse positives: {len(false_positives)} | False negatives: {len(false_negatives)}")
print("\nSample false positives (predicted risky, actually stable):")
print(false_positives[["content_id"] + features].head(5).to_string(index=False))
print("\nSample false negatives (predicted safe, actually declining):")
print(false_negatives[["content_id"] + features].head(5).to_string(index=False))

Feature importance (permutation):
               feature  importance
       impressions_90d    0.056848
      content_age_days    0.023323
          avg_position    0.013619
                   ctr    0.011978
       engagement_rate    0.001982
          sessions_90d    0.001705
            word_count   -0.007086
days_since_last_update   -0.013889

False positives: 1815 | False negatives: 1230

Sample false positives (predicted risky, actually stable):
          content_id  avg_position  impressions_90d  sessions_90d  content_age_days  days_since_last_update  ctr  engagement_rate  word_count
content_a5a2fbc76336          39.8              307             4               238                     103 0.00              0.0      1342.0
content_72c5c2d73e5a          30.0             2426             9               300                      13 0.12              0.0      2686.0
content_55f75c034970           6.4             3998             5               140                       8 0.03      

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.